# Notebook 33 — Final Locked-Test Evaluation

This notebook makes the final promotion decision for the experimental Corrected V4 + author-affinity policy. The configuration is frozen from Notebook 31:

```text
author_weight = 0.20
popularity_weight = 0.02
recency_tau = 3.0
```

No test metric is used to select or tune a parameter. The notebook:

1. downloads the rebuilt test split and locked protocol from Notebook 32;
2. reconstructs strict-prior corrected sequences only for the locked queries;
3. immediately persists that small corrected delta to Hugging Face for restart safety;
4. compares raw/matched Corrected V4, fixed Corrected V4 + author affinity, and Legacy V4.2 over one exact candidate set;
5. applies paired user-bootstrap and cold-item non-regression gates.

Use an **A100 High-RAM** runtime when possible. L4 High-RAM is the practical fallback.

In [ ]:
from google.colab import drive, files, userdata
drive.mount('/content/drive')

from getpass import getpass
from pathlib import Path
import hashlib, importlib.util, json, os, shutil, subprocess, sys, zipfile

DRIVE_ROOT = Path('/content/drive/MyDrive/recsys')
BASE_GOLD_ROOT = DRIVE_ROOT / 'data/gold/two_tower/v1_colab'
TRAIN_SOURCE = DRIVE_ROOT / 'data/gold/adaptive/v1/sequence_examples/train'
WORK_ROOT = Path('/content/recsys_corrected_v4_locked_test')
SOURCE_ROOT = WORK_ROOT / 'runtime'
HF_SNAPSHOT_ROOT = WORK_ROOT / 'hf_snapshot'
ADAPTIVE_ROOT = HF_SNAPSHOT_ROOT / 'adaptive_test/v1'
PROTOCOL_DIR = ADAPTIVE_ROOT / 'evaluation_protocol/v2_test'
CORRECTED_ROOT = ADAPTIVE_ROOT / 'corrected_queries/v1'
CORRECTED_V4_DIR = WORK_ROOT / 'artifacts/corrected_v4'
LEGACY_V4_DIR = WORK_ROOT / 'artifacts/legacy_v4'
REPORT_DIR = WORK_ROOT / 'reports/locked_test'
DRIVE_REPORT_DIR = DRIVE_ROOT / 'artifacts/adaptive/corrected_v4_locked_test_v1'
SPARK_LOCAL_DIR = Path('/content/spark-local')
for path in (SOURCE_ROOT, HF_SNAPSHOT_ROOT, CORRECTED_V4_DIR, LEGACY_V4_DIR, REPORT_DIR, DRIVE_REPORT_DIR, SPARK_LOCAL_DIR):
    path.mkdir(parents=True, exist_ok=True)
print('work root:', WORK_ROOT)

In [ ]:
required = {
    'torch': 'torch', 'pandas': 'pandas', 'pyarrow': 'pyarrow',
    'pyspark': 'pyspark==3.5.6', 'huggingface_hub': 'huggingface_hub>=0.32',
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
import torch
assert torch.cuda.is_available(), 'Select an A100 High-RAM or L4 High-RAM GPU runtime.'
free_gib = shutil.disk_usage('/content').free / 1024**3
ram_gib = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1024**3
assert free_gib >= 25, f'At least 25 GiB local disk is required; only {free_gib:.1f} GiB is free.'
assert ram_gib >= 20, f'A High-RAM runtime is required; only {ram_gib:.1f} GiB is available.'
print('GPU:', torch.cuda.get_device_name(0))
print(f'RAM: {ram_gib:.1f} GiB | local disk free: {free_gib:.1f} GiB')

In [ ]:
required_drive = [BASE_GOLD_ROOT / 'history/events', BASE_GOLD_ROOT / 'item_features/static', TRAIN_SOURCE]
missing_drive = [str(path) for path in required_drive if not path.exists()]
assert not missing_drive, 'Missing Drive inputs:\n' + '\n'.join(missing_drive)
print('Drive preflight: PASS')

## Download verified runtime, test data and checkpoints

Enable notebook access for both `HF_TOKEN` and `HF_WRITE_TOKEN`. The write token is used only to persist the corrected locked-query delta and compact final reports.

In [ ]:
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
from huggingface_hub.utils import EntryNotFoundError

HF_REPO_ID = 'khoa1505/recsys-artifacts'
RUNTIME_FILE = 'runtime/corrected_v4_locked_test_runtime_v1.zip'
RUNTIME_SHA256 = '9d0bc077ed9558e7f99b326f96bc82108cafbdd0d240fdfa02b8dad4793641b0'
CORRECTED_V4_FILE = 'v4_corrected/v1/best_model.pt'
LEGACY_V4_FILE = 'v4/v4_inference_fp16.pt'
LEGACY_V4_SHA256 = '4ad49b3e6f9f8b5d45a6c92f31d8d9b27f558e1746613efb44a23db3cd456a26'

def normalize_token(raw):
    if isinstance(raw, bytes): raw = raw.decode()
    if isinstance(raw, str):
        raw = raw.strip()
        return raw if raw.startswith('hf_') else None
    if isinstance(raw, dict):
        for value in raw.values():
            token = normalize_token(value)
            if token: return token
    return None

def secret(name):
    try: return normalize_token(userdata.get(name))
    except Exception: return None

HF_TOKEN = secret('HF_TOKEN') or normalize_token(getpass('Paste Hugging Face READ token: '))
HF_WRITE_TOKEN = secret('HF_WRITE_TOKEN') or normalize_token(getpass('Paste Hugging Face WRITE token: '))
assert HF_TOKEN and HF_WRITE_TOKEN
api = HfApi(token=HF_WRITE_TOKEN)
probe = WORK_ROOT / 'write_probe.txt'; probe.write_text('locked-test-write-probe\n')
api.upload_file(repo_id=HF_REPO_ID, repo_type='model', path_or_fileobj=str(probe), path_in_repo='adaptive_test/.locked_test_probe.txt', commit_message='Verify locked test write access')
api.delete_file(repo_id=HF_REPO_ID, repo_type='model', path_in_repo='adaptive_test/.locked_test_probe.txt', commit_message='Remove locked test write probe')

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        while block := stream.read(8 * 1024 * 1024): digest.update(block)
    return digest.hexdigest()

runtime_zip = Path(hf_hub_download(HF_REPO_ID, RUNTIME_FILE, repo_type='model', token=HF_TOKEN))
corrected_v4 = Path(hf_hub_download(HF_REPO_ID, CORRECTED_V4_FILE, repo_type='model', token=HF_TOKEN))
legacy_v4 = Path(hf_hub_download(HF_REPO_ID, LEGACY_V4_FILE, repo_type='model', token=HF_TOKEN))
assert sha256(runtime_zip) == RUNTIME_SHA256
assert sha256(legacy_v4) == LEGACY_V4_SHA256
assert corrected_v4.stat().st_size > 100_000_000
snapshot_download(
    HF_REPO_ID, repo_type='model', token=HF_TOKEN, local_dir=HF_SNAPSHOT_ROOT,
    allow_patterns=['adaptive_test/v1/manifest.json', 'adaptive_test/v1/sequence_examples/test/**', 'adaptive_test/v1/evaluation_protocol/v2_test/**'],
)
for source, destination in ((corrected_v4, CORRECTED_V4_DIR / 'best_model.pt'), (legacy_v4, LEGACY_V4_DIR / 'best_model.pt')):
    destination.unlink(missing_ok=True); destination.symlink_to(source.resolve())
test_manifest = json.loads((ADAPTIVE_ROOT / 'manifest.json').read_text())
protocol_manifest = json.loads((PROTOCOL_DIR / 'manifest.json').read_text())
assert test_manifest['quality']['passed'] is True
assert protocol_manifest['split'] == 'test' and protocol_manifest['queries'] == 11737
print('runtime/test/checkpoint download: PASS')

In [ ]:
if SOURCE_ROOT.exists(): shutil.rmtree(SOURCE_ROOT)
SOURCE_ROOT.mkdir(parents=True)
with zipfile.ZipFile(runtime_zip) as archive:
    assert archive.testzip() is None
    archive.extractall(SOURCE_ROOT)
SEQUENCE_SCRIPT = SOURCE_ROOT / 'scripts/build_corrected_test_query_sequences.py'
EVAL_SCRIPT = SOURCE_ROOT / 'scripts/evaluate_corrected_v4_author_affinity_test.py'
assert SEQUENCE_SCRIPT.exists() and EVAL_SCRIPT.exists()
compile(SEQUENCE_SCRIPT.read_text(), str(SEQUENCE_SCRIPT), 'exec')
compile(EVAL_SCRIPT.read_text(), str(EVAL_SCRIPT), 'exec')
print('runtime extraction/compile: PASS')

## Build or restore corrected locked-query sequences

The first run replays canonical history with timestamp-batch semantics. Once its quality gates pass, the result is uploaded before GPU evaluation. Future reruns restore it directly.

In [ ]:
remote_corrected_manifest = 'adaptive_test/v1/corrected_queries/v1/manifest.json'
if api.file_exists(HF_REPO_ID, remote_corrected_manifest, repo_type='model'):
    snapshot_download(
        HF_REPO_ID, repo_type='model', token=HF_TOKEN, local_dir=HF_SNAPSHOT_ROOT,
        allow_patterns=['adaptive_test/v1/corrected_queries/v1/**'],
    )
    print('restored corrected locked-query delta from Hugging Face')
else:
    shutil.rmtree(CORRECTED_ROOT, ignore_errors=True)
    env = os.environ.copy(); env['SPARK_MASTER'] = 'local[2]'; env['SPARK_LOCAL_DIR'] = str(SPARK_LOCAL_DIR)
    env['PYSPARK_SUBMIT_ARGS'] = '--driver-memory 14g pyspark-shell'
    command = [
        sys.executable, str(SEQUENCE_SCRIPT), '--base-gold-root', str(BASE_GOLD_ROOT),
        '--protocol-dir', str(PROTOCOL_DIR), '--output-dir', str(CORRECTED_ROOT),
        '--max-sequence-len', '30', '--shuffle-partitions', '64',
        '--output-partitions', '8', '--overwrite',
    ]
    sequence_log = Path('/content/corrected_test_sequences.log')
    process = subprocess.Popen(command, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    with sequence_log.open('w') as log:
        for line in process.stdout:
            print(line, end=''); log.write(line); log.flush()
    code = process.wait()
    if code != 0:
        tail = '\n'.join(sequence_log.read_text(errors='replace').splitlines()[-240:])
        raise RuntimeError(f'Corrected test sequence build failed with exit code {code}:\n{tail}')
    corrected_manifest = json.loads((CORRECTED_ROOT / 'manifest.json').read_text())
    assert corrected_manifest['quality']['passed'] is True
    api.upload_folder(
        repo_id=HF_REPO_ID, repo_type='model', folder_path=str(CORRECTED_ROOT),
        path_in_repo='adaptive_test/v1/corrected_queries/v1',
        commit_message='Upload corrected locked test query sequences',
    )
    assert api.file_exists(HF_REPO_ID, remote_corrected_manifest, repo_type='model')
    print('corrected locked-query delta persisted: PASS')
corrected_manifest = json.loads((CORRECTED_ROOT / 'manifest.json').read_text())
assert corrected_manifest['quality']['passed'] is True
print('corrected sequence quality:', corrected_manifest['quality'])

## Final exact locked-test evaluation

The weights below are deliberately literals, not a search grid. All variants use identical queries, exact targets, candidate eligibility, prior-seen filtering and candidate item order.

In [ ]:
REPORT_DIR.mkdir(parents=True, exist_ok=True)
command = [
    sys.executable, str(EVAL_SCRIPT), '--adaptive-root', str(ADAPTIVE_ROOT),
    '--train-source', str(TRAIN_SOURCE), '--base-gold-root', str(BASE_GOLD_ROOT),
    '--corrected-root', str(CORRECTED_ROOT), '--corrected-v4-dir', str(CORRECTED_V4_DIR),
    '--legacy-v4-dir', str(LEGACY_V4_DIR), '--protocol-dir', str(PROTOCOL_DIR),
    '--output-dir', str(REPORT_DIR), '--author-weight', '0.2',
    '--popularity-weight', '0.02', '--recency-tau', '3.0',
    '--query-batch-size', '16', '--item-batch-size', '8192',
    '--bootstrap-samples', '5000', '--seed', '71', '--device', 'cuda',
]
eval_log = Path('/content/corrected_v4_locked_test.log')
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
with eval_log.open('w') as log:
    for line in process.stdout:
        print(line, end=''); log.write(line); log.flush()
code = process.wait()
if code != 0:
    tail = '\n'.join(eval_log.read_text(errors='replace').splitlines()[-260:])
    raise RuntimeError(f'Locked test evaluation failed with exit code {code}:\n{tail}')
REPORT_JSON = REPORT_DIR / 'locked_test_evaluation.json'
assert REPORT_JSON.exists()
report = json.loads(REPORT_JSON.read_text())
print('locked test evaluation: PASS')

In [ ]:
import pandas as pd
print('frozen config:', report['fixed_config'])
print('acceptance gates:', json.dumps(report['acceptance_gates'], indent=2))
display(pd.read_csv(REPORT_DIR / 'test_summary.csv'))
display(pd.read_csv(REPORT_DIR / 'test_segments.csv').query("segment_type == 'target_warm'"))
for comparison, values in report['paired_user_bootstrap'].items():
    print('\n', comparison, 'HR@1000:', values['hitrate@1000'])

In [ ]:
small_report = WORK_ROOT / 'small_report'
shutil.rmtree(small_report, ignore_errors=True); small_report.mkdir()
for name in ('locked_test_evaluation.json', 'test_summary.csv', 'test_segments.csv'):
    shutil.copy2(REPORT_DIR / name, small_report / name)
shutil.copy2(eval_log, small_report / eval_log.name)
shutil.copy2(CORRECTED_ROOT / 'manifest.json', small_report / 'corrected_test_sequence_manifest.json')
sequence_log_path = Path('/content/corrected_test_sequences.log')
if sequence_log_path.exists(): shutil.copy2(sequence_log_path, small_report / sequence_log_path.name)
api.upload_folder(
    repo_id=HF_REPO_ID, repo_type='model', folder_path=str(small_report),
    path_in_repo='adaptive_test/v1/final_evaluation', commit_message='Upload final locked test evaluation',
)
for path in small_report.iterdir(): shutil.copy2(path, DRIVE_REPORT_DIR / path.name)
archive = Path(shutil.make_archive('/content/corrected_v4_locked_test_v1_report', 'zip', small_report))
print('Drive report:', DRIVE_REPORT_DIR)
print('download:', archive, f'{archive.stat().st_size / 1024**2:.2f} MiB')
files.download(str(archive))
print('FINAL LOCKED-TEST DECISION:', 'PROMOTE' if report['accepted'] else 'DO NOT PROMOTE')